# Feature Engineering for Philippine Soil Crop Suitability
# --------------------------------------------------------------------------------
# This notebook transforms raw soil and environmental data into a set of agronomic 
# indicators. Feature engineering is critical in agricultural modeling as raw 
# chemical measurements (e.g., ppm of Phosphorus) are often less predictive 
# than their interpreted classes (e.g., "Low" vs "High").
# 
# The engineering process follows six primary blocks:
# 1. **Missingness Encoding**: Capturing the systemic nature of lab test absences.
# 2. **Agronomic Interpretation**: Translating chemistry into BSWM ratings.
# 3. **Structural Health**: Analyzing soil compaction and rooting depth.
# 4. **Agroclimatic Interactions**: Combining climate, elevation, and season.
# 5. **Farm History Dynamics**: Encoding the impact of prior land use.
# 6. **Statistical Normalization**: Applying transforms to skewed distributions.
# --------------------------------------------------------------------------------

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src directory to path to import the pipeline
sys.path.append(os.path.abspath('../src'))
from pipeline import SoilDataPipeline, SoilFeatureEngineer

# Load data
df = pd.read_csv('../data/raw/ph_soil_crop.csv')

# Initialize and run preprocessing
pipeline = SoilDataPipeline()
df_cleaned = pipeline.preprocess(df)

# Initialize and run feature engineering
engineer = SoilFeatureEngineer()
df_engineered = engineer.transform(df_cleaned)

print(f"Original features: {df.shape[1]}")
print(f"Engineered features: {df_engineered.shape[1]}")
df_engineered.head()

### Block 1: Missingness Encoding
In soil science, the absence of a test is often a feature in itself. For example, high-cost tests like CEC are frequently skipped in remote regions. By creating binary flags for missing values, the model can learn if the "missingness" correlates with specific regions or soil types.

In [ ]:
# Display the missingness flags created by the pipeline
missing_flags = [col for col in df_engineered.columns if '_missing' in col]
display(df_engineered[missing_flags].sum())

### Block 2: Agronomic Interpretation (BSWM Ratings)
Raw values for N, P, and K are difficult for models to use directly because the "ideal" range changes by crop. We map these to "Low", "Medium", and "High" categories based on Bureau of Soils and Water Management (BSWM) standards.

We also calculate a **Fertility Index**, a composite score that represents the overall nutrient health of the sample.

In [ ]:
# Analyzing the distribution of the Fertility Index
plt.figure(figsize=(10, 6))
sns.histplot(df_engineered['fertility_index'], kde=True, color='navy')
plt.title('Distribution of Global Soil Fertility Index')
plt.xlabel('Fertility Index (0-1)')
plt.ylabel('Frequency')
plt.show()

# Distribution of N, P, K ratings
ratings_cols = ['n_rating', 'p_rating', 'k_rating']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, col in enumerate(ratings_cols):
    sns.countplot(data=df_engineered, x=col, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

### Block 3: Soil Health and Structural Features
Physical properties like bulk density and rooting depth determine the "habitability" of the soil. We derive binary flags for compaction and soil depth adequacy, and an ordinal scale for drainage class.

**Concept:** A soil can be chemically rich (High NPK) but physically unusable (Severely Compacted), which severely penalizes the suitability of root-sensitive crops.

In [ ]:
# Inspecting structural features
structural_cols = ['is_compacted', 'shallow_soil', 'deep_soil', 'drainage_ordinal', 'om_adequate']
display(df_engineered[structural_cols].value_counts().head())

# Distribution of Drainage Ordinal
plt.figure(figsize=(8, 6))
sns.countplot(data=df_engineered, x='drainage_ordinal', palette='magma')
plt.title('Distribution of Soil Drainage (1:Poorly to 5:Excessive)')
plt.show()

### Block 4: Climate and Seasonal Interactions
Agriculture is an interaction between soil and atmosphere. We create "Agroclimatic Zones" by combining climate type with elevation bins. We also flag specific environmental conditions like "Highland Zones" (cool/high) and "Irrigated Potential" (flat/water-proximate).

In [ ]:
# Analyzing Agroclimatic Zones
zone_counts = df_engineered['agroclimatic_zone'].value_counts()
plt.figure(figsize=(12, 8))
sns.barplot(x=zone_counts.values, y=zone_counts.index, palette='viridis')
plt.title('Distribution of Agroclimatic Zones (Climate + Elevation)')
plt.show()

# Checking for Highland and Irrigated potentials
interaction_cols = ['peak_growing_period', 'highland_zone', 'irrigated_potential', 'lowland_heat', 'highland_cool']
display(df_engineered[interaction_cols].sum())

### Block 5: Farm History Dynamics
Soil fertility is a temporal process. Monocropping depletes nutrients, while rotation or virgin land provides a fertility bonus. We encode these as a "Soil Depletion Risk" flag and a "Farm History Score".

In [ ]:
# Validating Farm History Scores
plt.figure(figsize=(8, 6))
sns.countplot(data=df_engineered, x='farm_history_score', palette='Set3')
plt.title('Distribution of Farm History Fertility Scores')
plt.show()

# Prior legume effect on Nitrogen (expected correlation)
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_engineered, x='prior_legume', y='nitrogen_pct', palette='coolwarm')
plt.title('Nitrogen Content: Prior Legume vs Non-Legume')
plt.show()

### Block 6: Statistical Normalization and Log-Transforms
Soil chemicals like Phosphorus often exhibit heavily right-skewed distributions. We apply $\log(1+x)$ transforms to these variables to stabilize variance and make the features more amenable to linear-based learners and neural networks.

In [ ]:
# Comparing original vs log-transformed phosphorus
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.histplot(df_engineered['phosphorus_ppm'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Original Phosphorus (PPM)')

sns.histplot(df_engineered['log_phosphorus'], kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Log-Transformed Phosphorus')

plt.show()

### Final Summary of Feature Engineering
We have expanded the original feature set through a combination of domain-specific agronomic rules and statistical transforms.

**Transformation Summary:**
- **Dimensionality**: Increased from 52 to ~80 features.
- **Information Gain**: Captured missingness patterns, interpreted chemistry into ratings, and created high-level environmental zones.
- **Stability**: Log-transforms were used to handle skewed distributions of NPK and elevation.

The dataset is now fully prepared for the regression and ranking stages of the ML pipeline.